# Notebook 3 — Train / Validation / Test Split

This notebook splits the labeled order-level dataset into training,
validation, and test sets.

The split is performed before detailed exploratory data analysis to
prevent information from the test set from influencing modeling decisions.

The labeled dataset created in Notebook 2 is used as the input artifact.

In [1]:
import pandas as pd

## 1. Load the Labeled Dataset

The labeled order-level dataset from Notebook 2 is loaded as the input
artifact for this notebook.

In [2]:
labeled_data = pd.read_csv(
    "artifacts/labeled_orders.csv"
)

print("Shape:", labeled_data.shape)
print("Unique orders:", labeled_data["order_id"].nunique())

Shape: (99441, 33)
Unique orders: 99441


In [3]:
labeled_data["order_purchase_timestamp"] = pd.to_datetime(
    labeled_data["order_purchase_timestamp"]
)

print("Earliest order:", labeled_data["order_purchase_timestamp"].min())
print("Latest order:", labeled_data["order_purchase_timestamp"].max())

Earliest order: 2016-09-04 21:15:19
Latest order: 2018-10-17 17:30:18


In [4]:
print(
    labeled_data["is_late"]
    .value_counts(normalize=True)
)

is_late
0    0.92129
1    0.07871
Name: proportion, dtype: float64


In [5]:
from sklearn.model_selection import train_test_split

In [6]:
train_data, temp_data = train_test_split(
    labeled_data,
    test_size=0.30,
    stratify=labeled_data["is_late"],
    random_state=42
)

In [7]:
validation_data, test_data = train_test_split(
    temp_data,
    test_size=0.50,
    stratify=temp_data["is_late"],
    random_state=42
)

In [8]:
print("Train shape:", train_data.shape)
print("Validation shape:", validation_data.shape)
print("Test shape:", test_data.shape)

Train shape: (69608, 33)
Validation shape: (14916, 33)
Test shape: (14917, 33)


In [9]:
print("Train label distribution:")
print(train_data["is_late"].value_counts(normalize=True))

print("\nValidation label distribution:")
print(validation_data["is_late"].value_counts(normalize=True))

print("\nTest label distribution:")
print(test_data["is_late"].value_counts(normalize=True))

Train label distribution:
is_late
0    0.921288
1    0.078712
Name: proportion, dtype: float64

Validation label distribution:
is_late
0    0.921293
1    0.078707
Name: proportion, dtype: float64

Test label distribution:
is_late
0    0.921298
1    0.078702
Name: proportion, dtype: float64


## 2. Split Strategy

A stratified random split was selected for the initial modeling workflow.

The dataset contains orders from September 2016 through October 2018.
A random split was chosen so that the training, validation, and test sets
contain similar proportions of late and on-time deliveries.

Stratification is important because late deliveries represent only about
7.87% of the dataset. Without stratification, the class proportions could
differ between the splits.

A fixed random state of 42 is used to make the split reproducible.

In [10]:
train_ids = set(train_data["order_id"])
validation_ids = set(validation_data["order_id"])
test_ids = set(test_data["order_id"])

print("Train ∩ Validation:", len(train_ids & validation_ids))
print("Train ∩ Test:", len(train_ids & test_ids))
print("Validation ∩ Test:", len(validation_ids & test_ids))

Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


In [11]:
all_split_ids = train_ids | validation_ids | test_ids

print("Total unique orders across splits:", len(all_split_ids))
print("Original unique orders:", labeled_data["order_id"].nunique())

Total unique orders across splits: 99441
Original unique orders: 99441


## 3. Save the Split Artifacts

The training, validation, and test datasets are saved as separate
artifacts for the next stages of the workflow.

The test set will remain untouched until final model evaluation.

In [12]:
import os

os.makedirs("artifacts", exist_ok=True)

train_data.to_csv(
    "artifacts/train.csv",
    index=False
)

validation_data.to_csv(
    "artifacts/validation.csv",
    index=False
)

test_data.to_csv(
    "artifacts/test.csv",
    index=False
)

print("Train, validation, and test artifacts saved successfully!")

Train, validation, and test artifacts saved successfully!


In [13]:
import os

for file_name in ["train.csv", "validation.csv", "test.csv"]:
    path = os.path.join("artifacts", file_name)
    print(file_name, "exists:", os.path.exists(path))

train.csv exists: True
validation.csv exists: True
test.csv exists: True
